In [3]:
import os
import sys
from datetime import datetime
import itertools

from importlib import reload
current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)
import plotting
import pinns
import infinite
reload(plotting)
reload(pinns)
reload(infinite)
import numpy as np
import sympy as sp
import pandas as pd
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn
import torch.optim as optim
from pinns import  MLP, init_weights, CoefficientNet, pde_loss_inf, observation_loss_u, observation_loss_k, train_dual_network, build_models, set_seed,run_experiment_inf,build_models_KAN
from pinns import build_models
from infinite import analytical_solution_inf, coefficient_inf, source_term_inf, generate_dataset_inf, evaluate_model_inf
torch.set_default_dtype(torch.float32)
from plotting import plot_histories_comparison

set_seed(1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [5]:
 
# =========================================================
# Load data from the master budget-matching summary CSV
# =========================================================
csv_path = os.path.join("results", "summary_metrics.csv")
df = pd.read_csv(csv_path)

# Separate KAN and MLP data and sort by parameters/flops for clean line plots
df_kan = df[df["model_type"] == "KAN"].sort_values("parameters")
df_mlp = df[df["model_type"] == "MLP"].sort_values("parameters")

# =========================================================
# Plotting (1x2 Grid: Parameters vs Error & FLOPs vs Error)
# =========================================================
fig, axes = plt.subplots(
    1, 2,
    figsize=(8.0, 3.5),
    sharey=True
)

axis_color = "gray"

# ---------------------------------------------------------
# Subplot 0: Parameters vs Error
# ---------------------------------------------------------
axes[0].plot(
    df_kan["parameters"], df_kan["err_u_global"], 
    marker='o', color='#1f77b4', linewidth=1.5, markersize=5, label="KAN"
)
axes[0].plot(
    df_mlp["parameters"], df_mlp["err_u_global"], 
    marker='s', color='#797979', linewidth=1.5, markersize=5, label="MLP"
)
axes[0].set_xlabel("Number of Parameters", fontsize=9)
axes[0].set_ylabel("Global Error ($u$)", fontsize=9)
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].legend(frameon=False, fontsize=8, loc="upper right")

# ---------------------------------------------------------
# Subplot 1: FLOPs vs Error
# ---------------------------------------------------------
axes[1].plot(
    df_kan["flops"], df_kan["err_u_global"], 
    marker='o', color='#1f77b4', linewidth=1.5, markersize=5, label="KAN"
)
axes[1].plot(
    df_mlp["flops"], df_mlp["err_u_global"], 
    marker='s', color='#797979', linewidth=1.5, markersize=5, label="MLP"
)
axes[1].set_xlabel("FLOPs", fontsize=9)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].legend(frameon=False, fontsize=8, loc="upper right")

# =========================================================
# Styling and Formatting
# =========================================================
for ax in axes:
    ax.tick_params(axis="both", labelsize=7, colors=axis_color)
    ax.grid(True, linestyle='--', alpha=0.4)
    for spine in ax.spines.values():
        spine.set_color(axis_color)
        spine.set_linewidth(0.5)

plt.subplots_adjust(wspace=0.25)
os.makedirs("figures", exist_ok=True)
plt.savefig("figures/budget_matched_accuracy_scaling.svg", bbox_inches="tight", dpi=300)
plt.show()

KeyError: 'parameters'